# 02 · Ön İşleme

01 defterindeki EDA'da verilen kararları bütün veriye uyguluyoruz:

| Karar | Kanıt (01) |
|---|---|
| Kanallar: **DBZ + VEL + WIDTH** | D3 |
| **En alçak tarama** (0,5°) | D4 |
| **Son zaman adımı** | D5 |
| Sabit ölçekleme aralıkları (WIDTH 0–16) | D2 |
| `uint8` ile saklama | D6 |
| Boş pikselleri doldurma | D1 |

1. Bölüm ön işlemenin ne yaptığını tek bir örnekte gösteriyor. 2. Bölüm seçilen yılların hepsini işleyip Google Drive'a kaydediyor.

**Colab'da CPU çalışma zamanıyla çalıştır** (GPU gerekmez).

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO)

In [ ]:
import shutil
import numpy as np
import matplotlib.pyplot as plt
from src import paths, tornet, viz

cfg = paths.load_config()
print("ayarlar:", cfg)

## 1. Ön işleme ne yapıyor? (tek örnekte gösterim)

Bütün veriye uygulamadan önce bir örnekte görelim ve kontrol edelim. Adımlar (`src/tornet.py`):

1. **Seçme:** DBZ, VEL ve WIDTH; en alçak tarama; son zaman adımı → boyut `(3, 120, 240)`.
2. **Ölçekleme:** Her kanal sabit bir fiziksel aralıkla [-1, 1]'e çevrilir (DBZ −20…60 dBZ, VEL −60…60 m/s, WIDTH 0…16 m/s). Kanallar farklı aralıklarda olursa ağ birine daha fazla önem verebilir ve öğrenme zorlaşır. Aralıklar sabit olduğu için test kümesinden bilgi sızmaz.
3. **Sıkıştırma:** Değerler 1–255 arası tam sayı (`uint8`) olarak saklanır; `0` boş pikseller için ayrılmıştır. Veri 4 kat küçülür.
4. **Boşlukları doldurma** (yüklerken): DBZ → −1 (yankı yok), VEL → 0 (hareket yok), WIDTH → −1 (türbülans yok).

In [ ]:
one = tornet.sample_from_tar(tornet.tar_url(2017), per_category={"TOR": 1})
ds = next(iter(one.values()))

q = tornet.preprocess_sample(ds, cfg["variables"], cfg["sweep"], cfg["frame"])   # saklanan hal
x = tornet.from_uint8(q, cfg["variables"])                                     # ağın girdisi
print("saklanan:", q.shape, q.dtype, "| ağın girdisi:", x.shape, x.dtype, f"[{x.min():.2f}, {x.max():.2f}]")
print("kanal başına boş piksel (%):", {v: round(float(p), 1) for v, p in zip(cfg["variables"], 100 * (q == 0).mean(axis=(1, 2)))})

**Yuvarlama kontrolü:** Saklanan değeri fiziksel birime geri çevirip ham değerle karşılaştırıyoruz. Hata, 01'deki D6'da hesapladığımız en büyük yuvarlama hatasını geçmemeli.

In [ ]:
for i, v in enumerate(cfg["variables"]):
    lo, hi = tornet.CHANNEL_RANGES[v]
    raw = ds[v].values[cfg["frame"], :, :, cfg["sweep"]]
    ok = np.isfinite(raw)
    back = (x[i][ok] + 1) / 2 * (hi - lo) + lo           # [-1, 1] -> fiziksel birim
    err = np.abs(np.clip(raw[ok], lo, hi) - back).max()
    print(f"{v:6s} en büyük hata: {err:.3f}  (sınır: {(hi - lo) / 254 / 2:.3f})")

**Ağın gördüğü görüntü.** Satırlar açı (0,5°'lik 120 hücre), sütunlar mesafe (250 m'lik 240 hücre). 01'deki haritaya göre şekiller gerilmiş görünür: yelpaze dilimi düz bir dikdörtgene açılmış gibi. Haritaya çevirmek pikseller arasında tahmin (enterpolasyon) gerektirir ve bilgi kaybettirir; bu yüzden ağa bu haliyle veriyoruz.

VEL'deki beyaz alanların bir kısmı gerçekten durgun hava, bir kısmı ölçüm olmayan yerler (0 ile doldurduk). D1'deki boş piksel oranlarını hatırla.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6), layout="constrained")
for ax, v, img in zip(axes, cfg["variables"], x):
    m = ax.imshow(img, aspect="auto", cmap=viz.CMAPS[v], vmin=-1, vmax=1)
    ax.set_title(f"{v} (ağın girdisi)")
    ax.set_xlabel("mesafe hücresi (250 m)")
    ax.set_ylabel("açı hücresi (0,5°)")
    plt.colorbar(m, ax=ax)
plt.show()

## 2. Seçilen yılların hepsini işleme

`configs/data.json` dosyasındaki her yıl için:

1. arşiv Zenodo'dan Colab diskine indirilir (15–18 GB, geçici),
2. her örnek **arşiv açılmadan** okunur ve yukarıdaki ön işleme uygulanır,
3. `tornet_<yıl>.npz` Google Drive'a kaydedilir (yıl başına ~1,5–2 GB),
4. arşiv silinir.

Biraz sürer (yıl başına ~15–30 dakika, çoğu indirme). Bağlantı koparsa defteri tekrar çalıştır: bitmiş yıllar atlanır, yarım kalan indirme kaldığı yerden devam eder.

Google Drive'da yaklaşık **6–7 GB** boş yer gerekir.

In [ ]:
SCRATCH = paths.scratch_dir()      # geçici, büyük
OUT = paths.processed_dir()        # kalıcı (Colab'da Drive)

def free_gb(p):
    return shutil.disk_usage(p).free / 1e9

print("yıllar:", cfg["years"], "| kanallar:", cfg["variables"])
print(f"geçici klasör: {SCRATCH}  ({free_gb(SCRATCH):.0f} GB boş)")
print(f"çıktı klasörü: {OUT}  ({free_gb(OUT):.0f} GB boş)")

In [ ]:
cat = tornet.get_catalog(paths.data_dir())

for year in cfg["years"]:
    out_file = OUT / f"tornet_{year}.npz"
    if out_file.exists():
        print(f"{year}: zaten tamamlanmış")
        continue
    tar_path = SCRATCH / f"tornet_{year}.tar.gz"
    print(f"{year}: indiriliyor ...")
    tornet.download(tornet.tar_url(year), tar_path)
    print(f"{year}: işleniyor ...")
    tornet.process_year(year, tar_path, OUT, cfg["variables"], cfg["sweep"], cfg["frame"], catalog=cat)
    tar_path.unlink()               # sonraki yıl için Colab diskinde yer aç
    print(f"{year}: arşiv silindi, geçici klasörde {free_gb(SCRATCH):.0f} GB boş")

## Sonuçları katalogla karşılaştır

In [ ]:
for year in cfg["years"]:
    d = np.load(OUT / f"tornet_{year}.npz")
    X, y, names = d["X"], d["y"], d["filename"]
    expected = cat[cat["year"] == year]
    match = cat.set_index("filename").loc[names, "label"].to_numpy()
    print(f"{year}: X {X.shape} {X.dtype} | kanallar {list(d['variables'])} | {len(y):,} örnek (katalog {len(expected):,}) | "
          f"{int(y.sum()):,} hortum | etiketler katalogla aynı: {(match == y).all()}")